# Revisao: Fit vs Transform
Notebook curto para visualizar treino e uso em novos dados.

In [ ]:
# Pandas para montar tabelas (DataFrame)
import pandas as pd

# Joblib para salvar e carregar o pipeline treinado
import joblib

# Path para criar pasta/arquivo de forma organizada
from pathlib import Path

# Ferramentas de transformacao
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
class MiniFeatureEngineer:
    # Classe pequena para ensinar fit, save, load e transform
    def __init__(self, model_dir="models"):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)
        self.path = self.model_dir / "mini_transformer.joblib"

        # Comeca vazio: sera preenchido no fit() ou load()
        self.pipeline = None
        self.feature_names = None

    def fit(self, df, num_cols, cat_cols):
        # "pre" (preprocessor) e o bloco que prepara os dados antes do modelo.
        # Ele aplica regras diferentes por tipo de coluna:
        # - numericas: MinMaxScaler
        # - categoricas: OneHotEncoder
        pre = ColumnTransformer(
            transformers=[
                ("num", MinMaxScaler(), num_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ]
        )

        # Pipeline com um passo de preprocessamento
        self.pipeline = Pipeline(steps=[("preprocessor", pre)])

        # fit = aprende min/max e categorias
        self.pipeline.fit(df)

        # Guarda os nomes finais das colunas para o DataFrame transformado
        cat = self.pipeline.named_steps["preprocessor"].named_transformers_["cat"]
        self.feature_names = num_cols + cat.get_feature_names_out(cat_cols).tolist()

    def transform(self, df):
        # Se nao houver pipeline em memoria, carrega do arquivo
        if self.pipeline is None:
            self.load()

        # transform = aplica regras aprendidas no fit, sem reaprender
        data = self.pipeline.transform(df)
        return pd.DataFrame(data, columns=self.feature_names)

    def save(self):
        # Salva tudo que foi aprendido no fit
        joblib.dump({"pipeline": self.pipeline, "feature_names": self.feature_names}, self.path)

    def load(self):
        # Recarrega o estado salvo para usar em producao
        state = joblib.load(self.path)
        self.pipeline = state["pipeline"]
        self.feature_names = state["feature_names"]

In [ ]:
# Dados de treino (onde o pipeline vai aprender)
df_train = pd.DataFrame([
    {"tempo_entrega": 22, "valor_pedido": 35, "tipo_entrega": "Moto"},
    {"tempo_entrega": 45, "valor_pedido": 90, "tipo_entrega": "Carro"},
    {"tempo_entrega": 30, "valor_pedido": 60, "tipo_entrega": "Bike"}
])

fe = MiniFeatureEngineer()

# fit aprende as regras do treino
fe.fit(df_train, ["tempo_entrega", "valor_pedido"], ["tipo_entrega"])

# salva para uso futuro
fe.save()

print("Transformer treinado e salvo com sucesso.")

In [ ]:
# Novos dados (simulando entrada da API)
df_novo = pd.DataFrame([
    {"tempo_entrega": 28, "valor_pedido": 50, "tipo_entrega": "Moto"},
    {"tempo_entrega": 38, "valor_pedido": 75, "tipo_entrega": "Patinete"}
])

# Em producao: carrega e apenas transforma
fe_prod = MiniFeatureEngineer()
fe_prod.load()
df_transformado = fe_prod.transform(df_novo)

df_transformado